In [1]:
import requests
import pandas as pd
from tqdm import tqdm



mappings = {
    '能源': [['国金证券'], ['石油行业','煤炭行业','采掘行业','燃气']],
    '有色金属': [['国金证券','华福证券'], ['有色金属','小金属','能源金属','贵金属']],
    '钢铁': [['华福证券'], ['钢铁行业','煤炭行业']],
    '化工': [['国金证券','国海证券'], ['化学制品','化纤行业','化学原料', '化肥行业', '塑料制品','橡胶制品', '非金属材料']],  
    
    '电力': [['国金证券'],['公用事业','电力行业','环保行业']], 
    '电力设备': [['国金证券','华福证券'], ['光伏设备','电池','风电设备','电网设备', '电源设备']], 
    
    '房地产': [['国金证券'],['房地产开发','房地产服务']], 
    '建材': [['国投证券','国信证券'], ['水泥建材','装修建材','玻璃玻纤']], 
    '建筑': [['国投证券'], ['工程建设','装修装饰']],  
    
    '金融': [['国金证券','华福证券'], ['银行', '证券','多元金融','保险']],
    '汽车': [['国金证券','华福证券'], ['汽车服务','汽车整车', '汽车零部件','交运设备']],
    '机械': [['国金证券'], ['专用设备', '通用设备','仪器仪表','工程机械']],
    '家电': [['国金证券','华福证券'], ['家电行业']],
    '电子': [['国金证券','华福证券'],['半导体', '消费电子','电子元件', '光学光电子','电子化学品']],
    
    '科技': [['国金证券','华福证券'], ['计算机设备',  '游戏', '通信设备', '通信服务', '互联网服务','软件开发','文化传媒']],
    '医药': [['国金证券','华福证券'], ['生物制品','医疗器械', '化学制药','医药商业','医疗服务','中药']],
    '食品饮料': [['国金证券','华福证券'], ['酿酒行业','食品饮料']],
    '农牧饲渔': [['国金证券'], ['农牧饲渔']],
    '轻工制造': [['华安证券','上海证券'], ['纺织服装', '家用轻工','造纸印刷', '包装材料']],
    '商贸零售': [['上海证券'], [ '商业百货', '美容护理', '珠宝首饰']],
    '社会服务': [['万联证券'], ['旅游酒店','教育','专业服务']], 
    '交通运输': [['国金证券','华福证券'], ['航运港口','航空机场', '物流行业','铁路公路']],
    
    '军工': [['中航证券'],['航天航空', '船舶制造']],
    
}


from datetime import datetime, timedelta
today = datetime.today()
lastweek_start = today - timedelta(days=today.weekday()+7)


def fetch_reports(page, beginTime, endTime):
    url = 'https://reportapi.eastmoney.com/report/list'
    payload ={
        'industryCode': '*',
        'pageSize': '100',
        'industry': '*',
        'rating': '*',
        'ratingChange': '*',
        'beginTime': beginTime.strftime("%Y-%m-%d"),
        'endTime': endTime.strftime("%Y-%m-%d"),
        'pageNo': str(page),
        'qType': '1'
    }

    r = requests.get(url, params = payload)
    total_page = r.json()['TotalPage']
    return r, total_page


total_df = pd.DataFrame()
_, total_page = fetch_reports(1, lastweek_start, today)

for page in tqdm(range(1,total_page+1)):
    r, _ = fetch_reports(page, lastweek_start, today)
    df = pd.DataFrame(r.json()['data'])[['title','orgSName','publishDate','infoCode','industryCode','industryName','emRatingValue','researcher']]
    df['publishDate'] = pd.to_datetime(df['publishDate']).dt.date.astype(str)
    total_df = pd.concat([total_df,df])


total_df.loc[total_df['title'].str.contains('能源周'), 'industryName'] = '石油行业'
df = pd.DataFrame(mappings).T
def format_title(x):
    try:
        if len(x.split('：')) > 1:
            x = ':'.join(x.split('：')[1:])
        else:
            x = x.split('：')[-1]
    except:
        pass   
    return x
    
df['title'] = '' 
df['infoCode'] = ''


for i, row in df.iterrows():
    tmp = total_df[total_df['orgSName'].isin(row[0]) & total_df['industryName'].isin(row[1])]
    tmp['title'] = tmp['title'].apply(format_title)
    df.at[i, 'title'] = (tmp['publishDate'] + ' || ' + tmp['title']).to_list()
    df.at[i, 'infoCode'] = tmp['infoCode'].to_list()

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:11<00:00,  1.00it/s]
C:\Users\Zhi\AppData\Local\Temp\ipykernel_2400\3766482918.py:92: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp['title'] = tmp['title'].apply(format_title)


In [3]:
from requests_html import HTMLSession

def fetch_highlight(infocode_list):
    highlight_list = []
    for infocode in infocode_list:
        url = f'https://data.eastmoney.com/report/zw_industry.jshtml?infocode={infocode}'
        session = HTMLSession()  
        r = session.get(url)  
        highlight = r.html.find('div.ctx-content')[0].text.replace('\n','').split('风险提示')[0]
        highlight_list.append(highlight)
        # pdf_link = list(r.html.find('a.pdf-link')[0].links)[0]
    return highlight_list
#使用tqdm进度条
tqdm.pandas() 
df['highlight'] = df['infoCode'].progress_map(fetch_highlight)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:53<00:00,  2.31s/it]


In [4]:
df

,0,1,title,infoCode,highlight
能源,[国金证券],"[石油行业, 煤炭行业, 采掘行业, 燃气]",[2024-11-12 || 美国大选对油价影响有限，维持基本面看跌观点],[AP202411121640828152],[原油油价展望：基于供需基本面，我们维持中期油价看空观点。特朗普上台同时带来了潜在利多和利空...
有色金属,"[国金证券, 华福证券]","[有色金属, 小金属, 能源金属, 贵金属]","[2024-11-11 || 特朗普赢得美国大选，美联储11月谨慎降息25BP, 2024-...","[AP202411111640811051, AP202411101640804615, A...",[投资要点：贵金属：特朗普赢得美国总统大选，美联储11月如期降息25BP。特朗普赢得美国大选...
钢铁,[华福证券],"[钢铁行业, 煤炭行业]","[2024-11-11 || 政策加持下焦煤配置价值逐步显现, 2024-11-10 || ...","[AP202411111640815658, AP202411101640804360, A...",[投资要点：动力煤截至2024年11月8日，秦港5500K动力末煤平仓价847元/吨，周环比...
化工,"[国金证券, 国海证券]","[化学制品, 化纤行业, 化学原料, 化肥行业, 塑料制品, 橡胶制品, 非金属材料]","[2024-11-13 || 轮胎三季报:东升西落趋势仍存，国产胎企开始分化, 2024-1...","[AP202411131640853466, AP202411121640839492, A...",[行业观点轮胎行业整体需求稳中向好，盈利端受到海运和原料影响有所承压。今年前三季度全球配套市...
电力,[国金证券],"[公用事业, 电力行业, 环保行业]","[2024-11-14 || 高温天气延续，居民负荷支撑用电, 2024-11-12 || ...","[AP202411141640878248, AP202411121640827359]",[拆分9M24用电数据——总量及分板块视角：1)9月全社会用电量8475.0亿千瓦时，一/二...
电力设备,"[国金证券, 华福证券]","[光伏设备, 电池, 风电设备, 电网设备, 电源设备]","[2024-11-11 || 大选情绪冲击完毕，关注点回归景气右侧信号与产业技术进步, 20...","[AP202411111640808584, AP202411111640806368, A...",[光伏&储能：美国大选“利空落地”，对A股相关标的股价造成压力的并非大选结果本身，而是美股光...
房地产,[国金证券],"[房地产开发, 房地产服务]","[2024-11-11 || 交易税率近期有望下调，11月新房供应减少, 2024-11-1...","[AP202411111640809290, AP202411111640807797, A...",[行业点评本周A股地产、港股物业上涨，港股地产下跌。本周（11.2-11.8）申万A股房地产...
建材,"[国投证券, 国信证券]","[水泥建材, 装修建材, 玻璃玻纤]",[],[],[]
建筑,[国投证券],"[工程建设, 装修装饰]",[],[],[]
金融,"[国金证券, 华福证券]","[银行, 证券, 多元金融, 保险]",[],[],[]


In [45]:
import requests
import os


def make_query(highlight_list):
    num = len(highlight_list)
    query = f'你是一名资深的投资顾问, 请分析以下{num}篇券商行业研报的摘要内容，然后总结行业的发展现状，并简述投资机会：'
    for i in highlight_list:
        query = query + f'摘要{highlight_list.index(i)+1}：{i}'
    return query

def ask_qwen(query):
    api_key = 'sk-7397a9299d73435cb10290493bddee4e'
    url = 'https://dashscope.aliyuncs.com/api/v1/services/aigc/text-generation/generation'
    headers = {'Content-Type': 'application/json',
               'Authorization':f'Bearer {api_key}'}
    body = {
        'model': 'qwen-long',
        "input": {
            "messages": [
                {"role": "system","content": "You are an experienced investment manager."},
                {"role": "user","content": query}
            ]
        },
        "parameters": {"result_format": "message"}
    }
    response = requests.post(url, headers=headers, json=body)
    try:
        answer = response.json()['output']['choices'][0]['message']['content']
    except:
        answer = response.json()
    return answer
    

In [22]:
qwen_df = df[['title','highlight']].rename(columns = {'title':'标题','highlight':'摘要'})
qwen_df['千问读研报'] = qwen_df['摘要'].apply(make_query).progress_map(ask_qwen)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [12:59<00:00, 33.87s/it]


In [44]:
a = qwen_df[qwen_df.index=='有色金属']
a['千问读研报'] = a['摘要'].apply(make_query).progress_map(ask_qwen)
print(a['千问读研报'][0])

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:59<00:00, 59.39s/it]

### 行业发展现状与投资机会总结

**行业发展现状：**
1. **贵金属**：特朗普赢得美国总统大选后，市场对关税和移民政策的担忧导致金价出现回调。美联储11月降息25BP，全球多个经济体的CPI低于2%的通胀目标，货币宽松趋势明显。短期内，美联储多次降息并未结束，全球政治不确定性继续支撑金价上涨。中长期看，美联储仍有降息空间，避险需求推动金价持续上涨。
2. **工业金属**：铜的供需关系紧张，供应端新增产能有限，需求端因终端客户的刚需释放和家电市场的回暖而表现良好。美联储降息和国内货币政策空间的打开将进一步提振铜价。铝方面，氧化铝价格持续上涨，电解铝企业因成本上升而减产，供需缺口逐渐扩大，预计铝价将迎来趋势性上涨。
3. **新能源金属**：锂市场全年过剩，但中长期看，锂矿为电动车产业链中最优质的标的，底部布局机会值得关注。稀土市场供需改善预期增强，管理条例的实施和“以旧换新”政策有望推升稀土价格。镍湿法冶炼项目利润可观，值得重点关注。

**投资机会：**
1. **贵金属**：建议关注黄金股如中金黄金、紫金矿业、山东黄金、赤峰黄金等，以及白银股如兴业银锡、银泰黄金、盛达资源等。这些公司在金价上涨周期中具有较高的弹性。
2. **工业金属**：
   - **铜**：推荐紫金矿业、洛阳钼业，关注铜陵有色、西部矿业、金诚信等。
   - **铝**：推荐中国铝业、中国宏桥，关注天山铝业、南山铝业、云铝股份、神火股份等。
3. **新能源金属**：
   - **锂**：建议关注盐湖股份、藏格矿业、永兴材料、中矿资源及紫金矿业，弹性关注江特电机、天齐锂业、赣锋锂业等。
   - **稀土**：建议关注北方稀土、金力永磁。
   - **镍**：关注力勤资源、华友钴业。
4. **其他小金属**：
   - **锑**：建议关注华锡有色、湖南黄金。
   - **钼**：建议关注金钼股份。
   - **锡**：建议关注华锡有色。
   - **钨**：建议关注钨精矿和仲钨酸铵相关公司。

### 详细分析

**摘要1**：
- **贵金属**：特朗普当选美国总统后，市场对关税和移民政策的担忧导致金价回调。美联储11月降息25BP，全球多个经济体的CPI低于2%的通胀目标，货币宽松趋势明显。短期内，美联储多次降息并未结束，全球政治不确定性继续支撑金价上涨。中


C:\Users\Zhi\AppData\Local\Temp\ipykernel_2400\688345335.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a['千问读研报'] = a['摘要'].apply(make_query).progress_map(ask_qwen)
C:\Users\Zhi\AppData\Local\Temp\ipykernel_2400\688345335.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(a['千问读研报'][0])


### 行业发展现状

#### 贵金属
- **短期**：特朗普赢得美国总统大选，市场担心关税和移民政策可能引发通胀，金价出现回调。美联储11月降息25BP，全球多个经济体CPI低于2%的通胀目标，货币宽松趋势明显。
- **中长期**：美联储降息空间仍在，政治经济不确定性带来避险需求，金价有望持续上涨。白银因其投资属性更强，弹性更大，表现可能优于黄金。

#### 工业金属
- **铜**：供给端受限，需求端因终端客户刚需释放和部分线缆企业促销政策而有所好转。美联储降息和中美贸易关系改善预期支撑铜价。中长期看，新能源需求强劲将带动供需缺口扩大，铜价有望继续上行。
- **铝**：氧化铝价格持续上行，部分电解铝企业因成本压力减产，铝锭继续去库。地产政策落地和新能源需求增长有望提振铝需求，铝价有望迎来趋势性上涨。

#### 新能源金属
- **锂**：2024年锂市场仍面临过剩局面，但中长期看，锂矿为电动车产业链最优质且弹性大的标的，建议关注底部战略性布局机会。
- **稀土**：稀土管理条例和“以旧换新”政策实施，供需改善预期增强，稀土价格有望上涨。当前稀土板块位于底部，建议关注相关标的。

#### 其他小金属
- **镍**：低成本镍湿法冶炼项目利润长存，建议关注相关标的。
- **锑**：缅甸局势不稳定导致供给减少，光伏玻璃“毒玻璃”事件可能规范行业生产，锑价有望迎来第二波上涨。
- **钼**：国内降息降准政策刺激下，钢厂盈利水平有望止跌回升，钼价上涨通道进一步明确。
- **锡**：缅甸锡矿复产进度不及预期，需求端受益于半导体复苏和光伏景气，锡供需格局长期向好。
- **钨**：价格和库存均有所上涨，供需格局稳定。

### 投资机会

#### 贵金属
- **黄金**：关注中金黄金、紫金矿业、山东黄金、赤峰黄金。低估弹性关注株冶集团和玉龙股份，其他关注银泰黄金、湖南黄金及招金矿业。
- **白银**：关注兴业银锡、银泰黄金、盛达资源。

#### 工业金属
- **铜**：推荐紫金矿业、洛阳钼业，小而优关注铜陵有色、西部矿业、金诚信，其他关注河钢股份、藏格矿业等。
- **铝**：推荐中国铝业、中国宏桥，关注天山铝业、南山铝业、云铝股份、神火股份等。

#### 新能源金属
- **锂**：关注盐湖股份、藏格矿业、永兴材料、中矿资源及紫金矿业；弹性

C:\Users\Zhi\AppData\Local\Temp\ipykernel_2400\2944666822.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(a['千问读研报'][0])


In [23]:
from IPython.display import display, HTML

def to_str(x):
    try:
        x = '<br>'.join(x)
    except:
        pass
    return x



# qwen_df['标题'] = qwen_df['标题'].apply(to_str)
qwen_df['千问读研报'] = qwen_df['千问读研报'].str.replace('\n','')
display(HTML(qwen_df[['标题','千问读研报']].to_html(escape=False, justify='left')))

,标题,千问读研报
能源,[2024-11-12 || 美国大选对油价影响有限，维持基本面看跌观点],### 行业发展现状分析#### 原油市场1. **供需基本面**： - **供给端**：OPEC+计划从2025年1月开始增产，尽管增产推迟一个月，但整体供应增加的预期依然强烈。 - **需求端**：国内原油进口量和加工量持续下滑，导致库存增加，短期内需求疲软。2025年国内刺激政策可能提振需求，但增长空间有限。 - **地缘政治**：中东地区近期相对平静，但未来局势仍存在不确定性，可能导致油价波动。2. **价格走势**： - 本周布伦特和WTI期货价格均小幅上涨，但中期看空观点不变。#### 天然气市场1. **供需基本面**： - **供给端**：美国天然气供应相对宽松，生产商可能随时复产，增量可达1亿立方米/天。欧洲方面，2025年俄罗斯管道气可能断流，存在约4000万立方米/天的下降空间。 - **需求端**：2024年12月和2025年2月气温可能同比下降，需求增加的可能性较大。2. **价格走势**： - 本周TTF和JKM价格均上涨，HH价格小幅上涨。未来1-2个月内市场偏空，但在2024年12月和2025年2月存在做多机会，尤其是TTF市场。#### 煤炭市场1. **供需基本面**： - **供给端**：2024年9月全国原煤产量同比增长4.40%，供应充足。 - **需求端**：截至2024年10月31日，重点电厂煤炭日耗量同比增长6.48%，需求稳定。2. **价格走势**： - 本周秦皇岛Q5500动力煤平仓价小幅下跌，整体市场较为平稳。### 推荐标的及理由#### 原油市场1. **标的**：中国石化（600028.SH） - **理由**：中国石化作为国内最大的炼油企业之一，具备较强的抗风险能力。尽管短期需求疲软，但2025年国内刺激政策有望提振需求，公司业绩有望改善。此外，公司在新能源领域也有布局，长期发展前景良好。2. **标的**：中国石油（601857.SH） - **理由**：中国石油作为国内最大的油气生产商，拥有丰富的资源储备和强大的供应链体系。尽管中期油价看空，但公司的多元化业务结构有助于分散风险。2025年OPEC+增产后，公司有望通过成本控制和效率提升保持竞争力。#### 天然气市场1. **标的**：新奥股份（600803.SH） - **理由**：新奥股份在天然气产业链中具有完整的布局，包括上游勘探、中游运输和下游销售。2024年12月和2025年2月的做多机会将对公司业绩产生积极影响。此外，公司在清洁能源领域的布局也为公司带来新的增长点。2. **标的**：昆仑能源（00135.HK） - **理由**：昆仑能源作为中国最大的城市燃气供应商之一，受益于国内天然气需求的稳步增长。公司在LNG接收站和管道运输方面的优势将进一步提升其市场地位。2025年俄罗斯管道气断流的风险也将增加欧洲市场的天然气需求，公司有望从中受益。#### 煤炭市场1. **标的**：中国神华（601088.SH） - **理由**：中国神华是国内最大的煤炭生产商之一，拥有丰富的煤炭资源和稳定的客户群体。尽管近期煤炭价格有所下跌，但公司的成本控制能力和市场占有率使其在竞争中占据优势。公司还在积极推进新能源转型，未来发展前景广阔。2. **标的**：兖矿能源（600188.SH） - **理由**：兖矿能源在煤炭生产和销售方面具有较强的实力，尤其是在高端煤种方面具备竞争优势。公司还积极拓展海外业务，降低单一市场风险。随着国内经济的逐步复苏，煤炭需求有望稳步增长，公司业绩有望持续改善。### 总结综合来看，原油市场短期内需求疲软，但2025年国内刺激政策有望提振需求；天然气市场在2024年12月和2025年2月存在做多机会，尤其是欧洲市场；煤炭市场供需基本平衡，价格较为平稳。推荐标的的选择主要基于公司的市场地位、业务布局和抗风险能力，以确保投资者在不同市场环境下都能获得稳健回报。
有色金属,"[2024-11-11 || 特朗普赢得美国大选，美联储11月谨慎降息25BP, 2024-11-10 || 氧化铝价持续上行，一体化铝企优势凸显, 2024-11-10 || 国内锑价逐步修复，关注供给扰动下的稀土涨价兑现, 2024-11-06 || 锑:出口恢复确立价格拐点，重视板块二轮布局机会, 2024-11-05 || 缅甸供给扰动超预期叠加“供改”整合，重视底部布局机会]",### 行业发展现状**1. 宏观环境与政策影响：**- **全球经济不确定性增加**：特朗普赢得美国总统大选，市场担心其政策可能导致通胀上升，全球多个经济体的CPI低于2%的通胀目标，货币宽松趋势明显。- **美联储降息**：11月美联储降息25BP，预计未来降息次数将增加，进一步推动全球货币宽松。- **政策支持**：国内出台多项政策支持消费和投资，包括增加地方化债资源、以旧换新政策等，有助于提振金属需求。**2. 金属价格走势：**- **贵金属**：金价短期受特朗普当选影响出现回调，但中长期看，避险需求和美联储降息预期将支撑金价上涨。白银因投资属性强，弹性更大。- **工业金属**：铜价受到供需紧平衡和美联储降息预期的支撑，预计中长期价格将上移。铝价因成本端氧化铝价格抬升和需求支撑，继续上行。- **新能源金属**：锂价短期内受供应过剩影响，但中长期看好锂矿作为电动车产业链的核心标的。稀土价格在供需改善和政策催化下有望上涨。**3. 供需关系：**- **铜**：供给端新增产能有限，需求端终端客户订单好转，但中小线缆企业订单一般。预计四季度铜价中枢上移。- **铝**：供给端新增产能和复产产能释放，但部分企业因成本高企减产。需求端受房地产政策和新能源需求支撑，预计铝价继续上涨。- **稀土**：缅甸局势不稳定导致进口减少，供需改善预期加强。《稀土管理条例》实施将进一步压缩供给，预计价格持续上涨。### 推荐标的及理由**1. 黄金和白银：**- **中金黄金**：作为国内领先的黄金生产企业，受益于金价上涨和避险需求增加。- **紫金矿业**：多元化金属矿产资源，黄金和铜业务均受益于价格上涨。- **山东黄金**：国内最大的黄金生产商之一，受益于金价上涨。- **银泰黄金**：拥有丰富的黄金和白银资源，受益于金银价格双重上涨。- **兴业银锡**：主要生产白银和锡，受益于白银和锡价格的上涨。**2. 铜：**- **紫金矿业**：多元化金属矿产资源，铜业务受益于价格上涨。- **洛阳钼业**：全球领先的铜生产商，受益于铜价上涨。- **铜陵有色**：国内大型铜生产企业，受益于铜价上涨。**3. 铝：**- **中国铝业**：国内最大的铝生产企业，受益于铝价上涨和供需改善。- **天山铝业**：具有完整产业链的铝生产企业，受益于铝价上涨。- **神火股份**：铝业务受益于铝价上涨和供需改善。**4. 稀土：**- **北方稀土**：国内最大的轻稀土生产企业，受益于稀土价格的上涨和政策支持。- **金力永磁**：高端磁材龙头企业，受益于稀土价格上涨和人形机器人的需求增长。- **中科三环**：国内领先的稀土永磁材料生产企业，受益于稀土价格上涨和新能源需求增长。**5. 锂：**- **盐湖股份**：国内最大的锂盐生产企业，受益于锂价上涨和新能源需求增长。- **赣锋锂业**：全球领先的锂生产商，受益于锂价上涨和新能源需求增长。- **天齐锂业**：全球最大的锂化合物生产商之一，受益于锂价上涨和新能源需求增长。### 总结当前金属行业受益于全球经济不确定性增加、美联储降息预期和政策支持，供需关系改善，价格有望持续上涨。推荐标的集中在黄金、铜、铝、稀土和锂等领域，这些企业在各自领域具有较强的竞争力和资源优势，有望在行业复苏中获得较好的收益。
钢铁,"[2024-11-11 || 政策加持下焦煤配置价值逐步显现, 2024-11-10 || 化债政策利在长远，市场逐渐回归基本面主导, 2024-11-04 || 供暖陆续开启产地先行上涨，关注后续增量政策, 2024-11-04 || 政策方向确认，钢铁基本面稳固，仍具备继续反弹基础]",### 行业发展现状#### 煤炭行业1. **价格走势**： - 动力煤价格在2024年11月初维持在847元/吨左右，周环比略有下跌，但随着北方天气转冷，供暖需求增加，预计Q4价格中枢将高于Q3。 - 焦煤价格有所下跌，但随着宏观政策的改善和财政政策的加码，焦煤价格无需过于悲观。2. **供需情况**： - 供应方面，陕蒙产地价格上涨，晋陕蒙三省煤矿开工率为83.5%，环比略有下降。焦煤开采难度加大，资源稀缺性显现。 - 需求方面，电厂日耗出现季节性下滑，但供暖需求增加，非电领域如甲醇和尿素开工率较高。3. **政策环境**： - 双碳政策背景下，产能控制严格，安监与环保政策趋严。 - 能源安全诉求下，煤炭或仍处于黄金时代，供给紧张态势持续。4. **市场前景**： - 煤炭作为主要能源的地位短期内难以改变，发电量持续增长，需求韧性较强。 - 在宏观经济偏弱、水电增多、新能源发展及进口煤增长的压力下，煤价仍维持高位震荡，企业利润可持续。#### 钢铁行业1. **价格走势**： - 钢材市场整体偏弱，螺纹钢价格小幅上涨，但整体波动不大。 - 原材料如铁矿石和焦炭价格波动较大，铁矿石到港量高位，港口库存高位运行，焦炭库存同比偏低。2. **供需情况**： - 供给方面，样本钢厂高炉和电炉开工率小幅下降，日均铁水产量减少。 

In [46]:
qwen_df.loc[qwen_df['标题'].apply(lambda x: True if x == [] else False), '千问读研报'] = ''

In [67]:
qwen_df['标题'] = qwen_df['标题'].str.replace('<br>','').str.replace('<br>','')

In [75]:
type(qwen_df['标题'][4])

C:\Users\Zhi\AppData\Local\Temp\ipykernel_2408\3518015777.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  type(qwen_df['标题'][4])


str

In [76]:

qwen_df[['标题','千问读研报']].to_csv('Qwen_reports.csv', index =  True)

In [67]:
import pywencai

query = '国家队持股'
loop = True
query_type = 'stock'
df = pywencai.get(query=query,loop = loop, log = True, query_type = query_type)
df.columns = [x.split('[')[0] for x in df.columns.tolist()]
df = df[['股票代码','股票简称','机构本期持股占流通股比例明细','持股机构名称明细']]#.rename(columns={"股票代码":"证券代码"})
df.columns = ['股票代码','股票简称','国家队持股比例','国家队机构名称']
   


[pywencai] 2024-11-15 22:33:31,338 - INFO - 获取condition开始
[pywencai] 2024-11-15 22:33:32,609 - INFO - 获取get_robot_data成功
[pywencai] 2024-11-15 22:33:32,611 - INFO - 第1页开始
[pywencai] 2024-11-15 22:33:33,771 - INFO - 第1页成功
[pywencai] 2024-11-15 22:33:33,775 - INFO - 第2页开始
[pywencai] 2024-11-15 22:33:34,909 - INFO - 第2页成功
[pywencai] 2024-11-15 22:33:34,913 - INFO - 第3页开始
[pywencai] 2024-11-15 22:33:35,797 - INFO - 第3页成功
[pywencai] 2024-11-15 22:33:35,801 - INFO - 第4页开始
[pywencai] 2024-11-15 22:33:36,716 - INFO - 第4页成功
[pywencai] 2024-11-15 22:33:36,720 - INFO - 第5页开始
[pywencai] 2024-11-15 22:33:37,687 - INFO - 第5页成功
[pywencai] 2024-11-15 22:33:37,691 - INFO - 第6页开始
[pywencai] 2024-11-15 22:33:38,641 - INFO - 第6页成功


In [74]:
GJD_dict={
'国家集成电路产业投资基金股份有限公司':'大基金一期',
'国家集成电路产业投资基金二期股份有限公司':'大基金二期',
'中国证券金融股份有限公司':'证金公司'
}
def clean_GJD(x):
    if '中央汇金' in x:
        x = '中央汇金'
    elif '基本养老保险' in x:
        x = '社保基金'
    else:
        for i in GJD_dict:
            x = x.replace(i,GJD_dict[i])
    return x 

In [75]:
df['国家队机构名称'] = df['国家队机构名称'].apply(clean_GJD)

In [77]:
df['国家队机构名称'].unique()

array(['中央汇金', '证金公司', '社保基金', '大基金一期', '大基金二期'], dtype=object)